# Tabular MLP and LSTM Fusion for Inference

### Introduction

In this notebook, we combine the models developed in **`Project.ipynb`** (LSTM) and **`tabular_pipeline_aeolus_mlp.ipynb`** (MLP) into a unified architecture. The resulting model jointly predicts both **arrival** and **departure delays** by leveraging complementary sources of information: **static tabular features** and **sequential flight-chain data**.

The MLP learns from aircraft, airport, operational, and weather-related tabular features, while the LSTM captures the temporal dependencies and delay propagation patterns within flight chains. By fusing the representations learned by these two models, the final architecture exploits both static and temporal information to improve flight delay prediction.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shap
import pickle
import copy
import os
import time
import shutil
import kagglehub

import torch
import torch.nn as nn
from torch.utils.data import Subset
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


### Data Preprocessing

Specify a specific year you want to analyze

In [ ]:
year = 2022

#### 1. Tabular Dataset

First, we prepare the Tabular Dataset, as done for MLP.

In [ ]:
import pyarrow.parquet as pq

def sample_parquet_streaming(path, frac=0.6, seed=42, batch_size=200_000):
    """Reads the parquet file in chunks, sampling each chunk, instead of
    loading the whole file and then calling .sample()."""
    rng = np.random.default_rng(seed)
    pf = pq.ParquetFile(path)
    chunks = []
    for batch in pf.iter_batches(batch_size=batch_size):
        chunk = batch.to_pandas()
        sampled = chunk.sample(frac=frac, random_state=rng.integers(0, 1_000_000))
        chunks.append(sampled)
    return pd.concat(chunks, ignore_index=True)


data_dir = "/content/drive/Othercomputers/local/Flight-Delay-Forecasting/data/tabular/2022/"

train_mlp = sample_parquet_streaming(data_dir + "train_encoded.parquet", frac=1)
val = pd.read_parquet(data_dir + "val_encoded.parquet")
test = pd.read_parquet(data_dir + "test_encoded.parquet")

target_col = "ARR_DELAY_BIN"
cat_cols = ["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_YEAR", "FL_MONTH", "FL_DAY", "FL_WEEK", "ORIGIN_INDEX", "DEST_INDEX"]
exclude_cols = {"ARR_DELAY", "DEP_DELAY", "ARR_DELAY_BIN", "DEP_DELAY_BIN", "_FLIGHT_DATE", "dep_hour_bucket"}
feature_cols = [c for c in train_mlp.columns if c not in exclude_cols]

print(train_mlp.shape, val.shape, test.shape)
print(f"Features used ({len(feature_cols)}): {feature_cols}")

(3776298, 38) (1282550, 38) (1282550, 38)
Features used (32): ['OP_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_YEAR', 'FL_MONTH', 'FL_DAY', 'ORIGIN_INDEX', 'DEST_INDEX', 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', 'CRS_ELAPSED_TIME', 'FLIGHTS', 'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD', 'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE', 'FL_WEEK', 'dep_hour_sin', 'dep_hour_cos', 'arr_hour_sin', 'arr_hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'great_circle_km', 'origin_congestion_2h']


In [ ]:
# Conversion into tensors

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

continuous_cols = [c for c in feature_cols if c not in cat_cols]

cont_mean = train_mlp[continuous_cols].mean()
cont_std = train_mlp[continuous_cols].std().replace(0, 1)

medians = train_mlp[continuous_cols].median()
train_mlp[continuous_cols] = train_mlp[continuous_cols].fillna(medians)
val[continuous_cols] = val[continuous_cols].fillna(medians)
test[continuous_cols] = test[continuous_cols].fillna(medians)

print("Missing remain in train:", train_mlp[continuous_cols].isna().sum().sum())

def normalize(df_):
    return ((df_[continuous_cols] - cont_mean) / cont_std).values.astype("float32")

cardinalities = {c: int(train_mlp[c].max()) + 2 for c in cat_cols}
def cat_array(df_):
    return np.stack([df_[c].clip(lower=-1).replace(-1, cardinalities[c]-1).values.astype("int64") for c in cat_cols], axis=1)

def make_tensors(df_):
    X_cat = torch.tensor(cat_array(df_), dtype=torch.long)
    X_cont = torch.tensor(normalize(df_), dtype=torch.float32)
    y = torch.tensor(df_[target_col].values, dtype=torch.float32)
    return X_cat, X_cont, y

X_cat_train, X_cont_train, y_train_mlp = make_tensors(train_mlp)
X_cat_val, X_cont_val, y_val_mlp = make_tensors(val)
X_cat_test, X_cont_test, y_test_mlp = make_tensors(test)

n_pos, n_neg = y_train_mlp.sum().item(), len(y_train_mlp) - y_train_mlp.sum().item()
pos_weight = torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32)

Device: cpu
Missing remain in train: 0


#### 2. Sequential Dataset

Then, we prepare the Tabular Dataset, as done for MLP.

In [ ]:
def load_dataset_pytorch(year, file_path="data/chain/"):

    split_types = ["train", "val", "test"]

    loaded_data = {}

    for split in split_types:
        full_file_path = file_path + f"{year}/{split}_flight_chain_{year}.pt"
        dataset = torch.load(full_file_path, weights_only=False)

        # Slice dense tensor to remove the last feature (FLIGHTS), which is always equal to 1
        dense = dataset.tensors[0]  # [N, seq_len, 7]
        dense = dense[
            :, :, :-1
        ].clone()  # now [N, seq_len, 6]. Clone to make sure not even the last column is copied into RAM
        # Rebuild dataset with the same other tensors
        loaded_data[split] = TensorDataset(dense, *dataset.tensors[1:])

        print(f"--- Read file: (split: {split}, year: {year}) ---")

    return loaded_data

In [ ]:
data_dir_chains = f"/content/drive/Othercomputers/local/Flight-Delay-Forecasting/data/chain/"

loaded_data=load_dataset_pytorch(year, file_path=data_dir_chains)

--- Read file: (split: train, year: 2022) ---
--- Read file: (split: val, year: 2022) ---
--- Read file: (split: test, year: 2022) ---


In [ ]:
# On targets
train_dataset = loaded_data["train"]
all_delays = train_dataset.tensors[4]      # [N, seq_len, 2]
all_valid_lens = train_dataset.tensors[3]  # [N]

seq_len = all_delays.shape[1]
mask = torch.arange(seq_len).unsqueeze(0) < all_valid_lens.unsqueeze(1)  # [N, seq_len]

valid_delays = all_delays[mask].float()  # [total_valid_steps, 2] -- only as big as valid entries, no dataset copy

delay_mean = valid_delays.mean(dim=0)  # shape [2]
delay_std = valid_delays.std(dim=0)    # shape [2]

print(f"Delay mean (ARR, DEP): {delay_mean}")
print(f"Delay std (ARR, DEP): {delay_std}")

# --- 2. Scale/unscale helpers (operate on tensors on-the-fly, no dataset copy) ---
def scale_tensors(targets, mean=delay_mean, std=delay_std):
    return (targets - mean.to(targets.device)) / std.to(targets.device)

def unscale_tensors(scaled, mean=delay_mean, std=delay_std):
    return scaled * std.to(scaled.device) + mean.to(scaled.device)

Delay mean (ARR, DEP): tensor([ 6.8322, 12.3763])
Delay std (ARR, DEP): tensor([54.0997, 52.0273])


### Architecture

The final architecture is obtained by combining the representations learned by the MLP and LSTM models. The MLP processes the static tabular features, while the LSTM encodes the sequential information contained in flight chains.


In [ ]:
all_sparse = train_dataset.tensors[1]  # [N, seq_len, 8]
sparse_cardinalities = [int(all_sparse[:, :, i].max().item()) + 1 for i in range(all_sparse.shape[-1])]
# maximum number of distinct values for each categorical (sparse) feature

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, cardinalities, cat_cols, n_continuous, hidden_dim=64,
                 embedding_dim=64):
        super().__init__()

        # Embedding layer for each categorical (sparse) feature.
        self.embeddings = nn.ModuleList([
            nn.Embedding(
                cardinalities[c], max(min(50, (cardinalities[c]+1)//2), 2))
            for c in cat_cols
        ])
        total_emb_dim = sum(e.embedding_dim for e in self.embeddings)

        # 1. MLP Feature Extractor
        """
        Input:
            [batch_size, total_embedding_dim + n_continuous]

        Output:
            [batch_size, embedding_dim]
        """

        self.mlp = nn.Sequential(
            nn.Linear(total_emb_dim + n_continuous, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, embedding_dim), nn.ReLU(),
        )

        # 2. Classification Head
        """
        Input:
            [batch_size, embedding_dim]

        Output:
            [batch_size, 1]
        """
        self.classifier_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, 1))

    def forward(self, x_cat, x_cont, return_embedding=False):

            embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
            x = torch.cat(embs + [x_cont], dim=1)
            embedding = self.mlp(x)

            if return_embedding:
                return embedding

            logit = self.classifier_head(embedding).squeeze(1)

            return logit

In [ ]:
class FlightChainLSTM(nn.Module):
    def __init__(self, dense_input_dim, sparse_cardinalities, embed_dim,
                 hidden_dim, dropout_prob, output_dim=2):
        super().__init__()

        # Embedding layer for each categorical (sparse) feature.
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=card, embedding_dim=embed_dim)
                for card in sparse_cardinalities
        ])

        total_sparse_embed_dim = embed_dim * len(sparse_cardinalities)

        self.total_input_dim = dense_input_dim + total_sparse_embed_dim
        self.hidden_dim = hidden_dim
        self.dropout_prob = dropout_prob


        # 1. LSTM Layer
        """
        Input: [batch_size, seq_len, total_input_dim]

        Output:
            lstm_out: [batch_size, seq_len, hidden_dim]
            hn: [num_layers, batch_size, hidden_dim]

            cn: [num_layers, batch_size, hidden_dim]
        """
        self.lstm = nn.LSTM(
            input_size=self.total_input_dim,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True
        )


        # Dropout layer applied to LSTM outputs for regularisation.
        self.dropout = nn.Dropout(self.dropout_prob)


        # 2. Fully Connected Regression Layer
        """
        Input: [batch_size, seq_len, hidden_dim]
        Output: [batch_size, seq_len, output_dim]

        where:
            output_dim = 2
            -> ARR_DELAY, DEP_DELAY
        """
        self.regressor = nn.Linear(
            self.hidden_dim,
            output_dim
        )


    def forward(self, dense_feat, sparse_feat, return_embedding=False):

        # Encode each sparse categorical feature independently.
        embedded = [
            emb(sparse_feat[:, :, i].long())
            for i, emb in enumerate(self.embeddings)
        ]


        sparse_embed = torch.cat(embedded, dim=-1)
        x = torch.cat((dense_feat, sparse_embed), dim=-1)


        # Learn temporal dependencies along the flight chain.
        lstm_out, (hn, cn) = self.lstm(x)


        # Apply dropout to LSTM representations.
        lstm_out = self.dropout(lstm_out)

        if return_embedding:
          return lstm_out

        # Predict arrival and departure delay for each flight.
        predictions = self.regressor(lstm_out)

        return predictions

In [ ]:
class FlightChainEarlyFusion(nn.Module):
    def __init__(self, mlp_module: nn.Module, dense_input_dim,
                 sparse_cardinalities, mlp_emb_dim, lstm_embed_dim, hidden_dim,
                 dropout_prob, output_dim=2):
        super().__init__()

        self.tabular_branch = mlp_module

        # Embedding layer for each categorical (sparse) feature..
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=card, embedding_dim=lstm_embed_dim)
                for card in sparse_cardinalities
        ])

        total_sparse_embed_dim = lstm_embed_dim * len(sparse_cardinalities)

        # LSTM receives:
        # dense features + sparse embeddings + MLP embedding
        self.total_input_dim = (
            dense_input_dim
            + total_sparse_embed_dim
            + mlp_emb_dim
        )

        self.lstm = nn.LSTM(
            input_size=self.total_input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout_prob)

        # Regression head
        """
        Input:
            [batch_size, seq_len, hidden_dim]

        Output:
            [batch_size, seq_len, 2] -> ARR_DELAY, DEP_DELAY
        """
        self.regressor = nn.Linear(
            hidden_dim,
            output_dim
        )


    def forward(self, mlp_x_cat, mlp_x_cont, lstm_dense, lstm_sparse):

        batch_size, seq_len, _ = mlp_x_cat.shape

        # MLP feature extraction
        mlp_x_cat_flat = mlp_x_cat.reshape(batch_size * seq_len, -1)
        mlp_x_cont_flat = mlp_x_cont.reshape(batch_size * seq_len, -1)

        tabular_embedding = self.tabular_branch(
            mlp_x_cat_flat,
            mlp_x_cont_flat,
            return_embedding=True
        )

        tabular_embedding = tabular_embedding.view(
            batch_size,
            seq_len,
            -1
        )

        # Sequential categorical embeddings
        embedded = [
            emb(lstm_sparse[:, :, i].long())
            for i, emb in enumerate(self.embeddings)
        ]

        sparse_embed = torch.cat(embedded, dim=-1)

        # Early fusion: combine all features before LSTM
        x = torch.cat([lstm_dense, sparse_embed, tabular_embedding], dim=-1)

        # forward and regression

        lstm_out, _ = self.lstm(x) # hn, cn not needed singularly
        lstm_out = self.dropout(lstm_out)
        predictions = self.regressor(lstm_out)

        return predictions

In [ ]:
class FlightChainLateFusion(nn.Module):
    def __init__(self, mlp_module: nn.Module, lstm_module: nn.Module,
                 mlp_emb_dim, lstm_hidden_dim, output_dim=2):
        super().__init__()

        self.tabular_branch = mlp_module
        self.sequential_branch = lstm_module

        fusion_dim = mlp_emb_dim + lstm_hidden_dim


        # Final Regression Layer
        self.regressor = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, output_dim)
        )


    def forward(self, mlp_x_cat, mlp_x_cont, lstm_dense,lstm_sparse):

        batch_size, seq_len, _ = mlp_x_cat.shape

        # Tabular branch
        mlp_x_cat_flat = mlp_x_cat.reshape(batch_size * seq_len, -1)
        mlp_x_cont_flat = mlp_x_cont.reshape(batch_size * seq_len, -1)

        tabular_embedding = self.tabular_branch(mlp_x_cat_flat, mlp_x_cont_flat,
            return_embedding=True)

        tabular_embedding = tabular_embedding.view(batch_size, seq_len, -1)

        # Sequential branch
        sequential_embedding = self.sequential_branch(
            lstm_dense,
            lstm_sparse,
            return_embedding=True
        )


        # Late fusion
        fused_features = torch.cat([
            tabular_embedding, sequential_embedding], dim=-1)

        # Regression
        predictions = self.regressor(fused_features)

        return predictions

#### Hyperparameters & Configuration

In [ ]:
DENSE_DIM = 6  # O_TEMP, D_TEMP, O_PRCP, D_PRCP, O_WSPD, D_WSPD.
# however no "FLIGHTS"
SPARSE_DIM = 8  # "MONTH","DAY_OF_WEEK","CRS_ARR_TIME_HOUR","CRS_DEP_TIME_HOUR",
#"ORIGIN_INDEX", "DEST_INDEX","OP_CARRIER", "OP_CARRIER_FL_NUM",
HIDDEN_UNITS = 32  # original 32
OUTPUT_REGRESSION_DIM = 2  # ARR_DELAY and DEP_DELAY

BATCH_SIZE = 64  # original 64
LEARNING_RATE = 0.001  # original 0.001
NUM_EPOCHS = 50  # original 50
PATIENCE = 10 # max number of iteration with no improvement in validation loss, for early stopping

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS=2

print(f"Using device: {DEVICE}")

# 2. Create DataLoaders from your filtered splits - NO: use original (already padded/truncated) dataset
# train_loader = DataLoader(filtered_loaded_data['train'], batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(filtered_loaded_data['val'], batch_size=BATCH_SIZE, shuffle=False)

fraction = 0.1  # for trial runs ---> about 500_000 samples

train_indices = torch.randperm(len(loaded_data["train"]))[
    : int(fraction * len(loaded_data["train"]))
]
val_indices = torch.randperm(len(loaded_data["val"]))[
    : int(fraction * len(loaded_data["val"]))
]
test_indices = torch.randperm(len(loaded_data["test"]))[
    : int(fraction * len(loaded_data["test"]))
]

train_loader = DataLoader(
    Subset(loaded_data["train"], train_indices),
    batch_size=BATCH_SIZE,
    shuffle=True,  # cant hurt in training
    num_workers=NUM_WORKERS,
    pin_memory=True,  # for faster but more memory consuming training
)
val_loader = DataLoader(
    Subset(loaded_data["val"], val_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,  # bc True unnecessary, would add computation overhead and lose reproducibility
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    Subset(loaded_data["test"], test_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

Using device: cpu


In [ ]:
# LSTM Model
lstm_model = FlightChainLSTM(
    dense_input_dim=DENSE_DIM,
    sparse_cardinalities=sparse_cardinalities,
    embed_dim=8,
    hidden_dim=HIDDEN_UNITS,
    output_dim=OUTPUT_REGRESSION_DIM
).to(DEVICE)


# MLP Model
mlp_model = TabularMLP(
    cardinalities=cardinalities,
    cat_cols=cat_cols,
    n_continuous=len(continuous_cols),
    hidden_dim=64,
    embedding_dim=64
).to(DEVICE)


# Fusion Model
fusion_model = FlightChainFusion(
    mlp_module=mlp_model,
    lstm_module=lstm_model,
    mlp_emb_dim=64,
    lstm_hidden_dim=HIDDEN_UNITS
).to(DEVICE)


print(fusion_model)

FlightChainFusion(
  (tabular_branch): TabularMLP(
    (embeddings): ModuleList(
      (0): Embedding(18, 9)
      (1): Embedding(6752, 50)
      (2): Embedding(2, 2)
      (3): Embedding(9, 5)
      (4): Embedding(32, 16)
      (5): Embedding(34, 17)
      (6): Embedding(317, 50)
      (7): Embedding(315, 50)
    )
    (mlp): Sequential(
      (0): Linear(in_features=223, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.4, inplace=False)
      (3): Linear(in_features=64, out_features=64, bias=True)
      (4): ReLU()
    )
    (classifier_head): Sequential(
      (0): Dropout(p=0.3, inplace=False)
      (1): Linear(in_features=64, out_features=1, bias=True)
    )
  )
  (sequential_branch): FlightChainLSTM(
    (embeddings): ModuleList(
      (0): Embedding(12, 8)
      (1): Embedding(7, 8)
      (2-3): 2 x Embedding(24, 8)
      (4-5): 2 x Embedding(322, 8)
      (6): Embedding(17, 8)
      (7): Embedding(6768, 8)
    )
    (lstm): LSTM(70, 32, batch_first=True)
   

In [ ]:
from torchview import draw_graph
import torch

graph = draw_graph(
    fusion_model,
    input_size=(
        (BATCH_SIZE, seq_len, len(cat_cols)),  # mlp_x_cat
        (BATCH_SIZE, seq_len, len(continuous_cols)), # mlp_x_cont
        (BATCH_SIZE, seq_len, DENSE_DIM),      # lstm_dense
        (BATCH_SIZE, seq_len, SPARSE_DIM)       # lstm_sparse
    ),
    dtypes=[
        torch.long, # mlp_x_cat
        torch.float, # mlp_x_cont
        torch.float, # lstm_dense
        torch.long # lstm_sparse
    ],
    expand_nested=True
)

graph.visual_graph.render("model_architecture", format="png")

'model_architecture.png'